## Train example generator without GO definitions + InterPro data

In [1]:
# Import necessary libraries
from datasets import load_dataset

# Load the CAFA5 dataset
print("Loading CAFA5 dataset...")
dataset = load_dataset(
    "wanglab/cafa5", 
    name="cafa5_reasoning"
)
train_dataset = dataset["train"]

# Load InterPro metadata
print("Loading InterPro metadata...")
ds_interpro = load_dataset(
    "wanglab/cafa5", 
    name="interpro_metadata"
)
interpro_metadata = ds_interpro['metadata'].to_pandas()

print(f"Dataset size: {len(train_dataset)}")
print(f"InterPro metadata size: {len(interpro_metadata)}")

# Convert to pandas for easier manipulation
train_df = train_dataset.to_pandas()

# Find a protein with InterPro annotations - FIXED VERSION
proteins_with_interpro = train_df[
    train_df['interpro_ids'].notna() & 
    (train_df['interpro_ids'].apply(lambda x: len(x) if x is not None else 0) > 0)
]
print(f"Found {len(proteins_with_interpro)} proteins with InterPro annotations")

Loading CAFA5 dataset...
Loading InterPro metadata...
Dataset size: 133538
InterPro metadata size: 36893
Found 126957 proteins with InterPro annotations


In [11]:
import random
from bioreason2.dataset.cafa5.processor import generate_cafa5_example

protein_id = 'P10321'  # Example protein with InterPro annotations
# protein_id = 'Q38G17'  # Example protein without InterPro annotations

interpro_metadata = ds_interpro['metadata'].to_pandas()
# interpro_metadata = None

if protein_id:
    protein_data = train_df[train_df['protein_id'] == protein_id]
    protein_data = protein_data.iloc[0]
else:
    # Random protein
    protein_data = train_df.iloc[random.randint(0, len(train_df)-1)]

example = generate_cafa5_example(
        protein_data,
        prompt_template=None,
        interpro_metadata=interpro_metadata,
        include_go_defs=False
)


has_interpro = (protein_data.get('interpro_ids') is not None and 
                len(protein_data.get('interpro_ids', [])) > 0)

print(f"Protein ID: {protein_data['protein_id']}")
print(f"Organism: {protein_data['organism']}")
print(f"Has InterPro: {'Yes' if has_interpro else 'No'}")

print("\n" + "-"*40 + " SYSTEM PROMPT " + "-"*40)
print(example['system'])

print("\n" + "-"*40 + " USER PROMPT " + "-"*40)
print(example['user'])

print("\n" + "-"*40 + " ASSISTANT REASONING " + "-"*40)
print(example['assistant_reasoning'])

print("\n" + "-"*40 + " ASSISTANT ANSWER " + "-"*40)
print(example['assistant_answer'])

Protein ID: P10321
Organism: Homo sapiens (Human)
Has InterPro: Yes

---------------------------------------- SYSTEM PROMPT ----------------------------------------
You are a scientific assistant specialized in protein function prediction. Given a protein sequence and organism information, analyze the protein and predict its functional annotations using Gene Ontology (GO) terms across three aspects: molecular functions (MF), biological processes (BP), and cellular components (CC). First predict InterPro terms for the protein as a list of entries using the tokens <|INTERPRO_SUMMARY_START|>/<|INTERPRO_SUMMARY_END|>. Each entry should contain InterPro entry ID, name, type, and span of protein sequence it is assigned to. Then for each GO aspect, systematically traverse the GO hierarchy from general root terms to specific leaf terms, predicting relevant child terms at progressively deeper levels. Organize your predictions by depth within each aspect and present them using the designated tok

### Previous version for comparison

In [9]:
from sft_data_generation_speedup import generate_training_example

example_prompt_template = {
        'system_prompt': "System: You are a scientific assistant specialized in protein function predictions. Given a protein's sequence embeddings and the organism it is associated with, predict the Gene Ontology (GO) terms describing its molecular functions (MF), biological processes this protein is involved in (BP), and cellular components where this protein is found (CC). Do this in a step-by-step manner, by traversing the GO graph from the root term of each subontology (depth 0) all the way to the leaf terms. At each depth, predict the child terms of each GO term that are associated with this protein. Once correct children GO terms are identified, repeat the same process for these GO terms at the next depth of the GO graph. After traversing the GO graph in each requested subontology, list all predicted GO terms, ordered by depth starting from the root. Finally, give a short summary of the protein's function based on the predicted GO terms.",
        
        'user_prompt': "User: Predict {aspect_str} GO terms for this protein:\nprotein sequence embeddings: <embeddings>\norganism: {organism}"
    }

protein_data = train_df[train_df['protein_id'] == protein_id].iloc[0]

training_example = generate_training_example(protein_data, example_prompt_template)

print('Protein ID:', protein_data.protein_id, "\n")
print(training_example)

Protein ID: P10321 

System: You are a scientific assistant specialized in protein function predictions. Given a protein's sequence embeddings and the organism it is associated with, predict the Gene Ontology (GO) terms describing its molecular functions (MF), biological processes this protein is involved in (BP), and cellular components where this protein is found (CC). Do this in a step-by-step manner, by traversing the GO graph from the root term of each subontology (depth 0) all the way to the leaf terms. At each depth, predict the child terms of each GO term that are associated with this protein. Once correct children GO terms are identified, repeat the same process for these GO terms at the next depth of the GO graph. After traversing the GO graph in each requested subontology, list all predicted GO terms, ordered by depth starting from the root. Finally, give a short summary of the protein's function based on the predicted GO terms.

User: Predict Molecular Function, Cellular Co